# Generative AI 004 - The Six LangChain Components

The lesson is a map: models, prompts, chains, indexes, memory, agents. Two of its
claims are stated as facts, and both can be checked here with **no API key**.

| Part | What we check |
|---|---|
| A | swapping provider changes **2 of 7 lines** |
| B | `ChatAnthropic` has 87 public members and defines **2** of them |
| C | what each memory strategy sends, turn by turn |
| D | buffer memory is **quadratic** - 16x the turns costs **291x** |

Part A needs nothing but the standard library. Parts B needs `langchain-core`
(and `langchain-anthropic` for the full table). Parts C and D need nothing.

## Part A - How much code really changes when you swap provider

The lesson says the difference between using OpenAI and using Anthropic is "one
or two lines". That is a diff, so measure it rather than believing it.

In [ ]:
import difflib

OPENAI = """from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
model = ChatOpenAI(model="gpt-4")
result = model.invoke("What is the capital of India?")
print(result.content)
"""

ANTHROPIC = """from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv

load_dotenv()
model = ChatAnthropic(model="claude-sonnet-4-5")
result = model.invoke("What is the capital of India?")
print(result.content)
"""

GOOGLE = """from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
result = model.invoke("What is the capital of India?")
print(result.content)
"""

In [ ]:
def changed(a, b):
    """Lines of b that differ from a, and the lines themselves."""
    al, bl = a.splitlines(), b.splitlines()
    sm = difflib.SequenceMatcher(None, al, bl)
    n, shown = 0, []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            n += max(i2 - i1, j2 - j1)
            shown += bl[j1:j2]
    return n, len(bl), shown


for name, a, b in [("OpenAI -> Anthropic", OPENAI, ANTHROPIC),
                   ("OpenAI -> Google", OPENAI, GOOGLE),
                   ("Anthropic -> Google", ANTHROPIC, GOOGLE)]:
    n, total, _ = changed(a, b)
    print(f"{name:<22} {n} of {total} lines change")

assert changed(OPENAI, ANTHROPIC)[0] == 2

In [ ]:
# Which two lines?
for line in changed(OPENAI, ANTHROPIC)[2]:
    print("  ", line)

# The import and the class name. load_dotenv, .invoke, .content and the
# print are untouched.

Two lines, and always the *same* two. That is what "model-agnostic development"
means when you cash it out.

## Part B - Why the diff is so small

It is not a coincidence and it is not politeness between competitors. Every chat
model inherits from one base class, so a provider's class is mostly not the
provider's code.

In [ ]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import Runnable
from langchain_core.language_models.fake_chat_models import FakeListChatModel


def public(cls):
    return {m for m in dir(cls) if not m.startswith("_")}


runnable, base = public(Runnable), public(BaseChatModel)
print(f"Runnable      {len(runnable):>4} public members")
print(f"BaseChatModel {len(base):>4} public members")
print(f"FakeListChatModel defines {len(public(FakeListChatModel) - base)} of its own")

In [ ]:
# Needs `pip install langchain-anthropic`. Skip this cell if you would rather not.
try:
    from langchain_anthropic import ChatAnthropic
    anth = public(ChatAnthropic)
    print(f"ChatAnthropic {len(anth):>4} public members")
    print(f"  inherited from BaseChatModel : {len(anth & base)}")
    print(f"  its own                      : {len(anth - base)}")
    print(f"  which are                    : {sorted(anth - base)}")
except ImportError:
    print("langchain-anthropic not installed - skipping")

Two of its own, out of eighty-seven. `invoke`, `stream`, `batch`, `bind_tools`
and the rest all come from the shared base.

**The numbers will drift** as the libraries change. The ratio is the point: a
provider class supplies connection details, not an interface.

## Part C - What memory costs

LLM API calls are **stateless**. Ask "Who is Narendra Modi?", then "How old is
he?", and the second call has no idea who "he" is. The fix is to send the earlier
conversation along with the new question - and the question is how much of it.

In [ ]:
# The conversation from the lesson, written out. Nothing is padded.
CONVERSATION = [
    ("Who is Narendra Modi?",
     "Narendra Modi is an Indian politician who has served as the Prime Minister of "
     "India since 2014. Before that he was the Chief Minister of Gujarat."),
    ("How old is he?",
     "He was born on 17 September 1950, which makes him 75 years old."),
    ("Which party does he belong to?",
     "He is a member of the Bharatiya Janata Party, usually shortened to the BJP."),
    ("Where was he born?",
     "He was born in Vadnagar, a small town in the Mehsana district of Gujarat."),
    ("What did he do before politics?",
     "He worked with the Rashtriya Swayamsevak Sangh from a young age and held "
     "organisational roles there before moving into electoral politics."),
    ("When did he first become Chief Minister?",
     "He became Chief Minister of Gujarat in October 2001 and held the post until "
     "2014, when he moved to national office."),
    ("How many terms has he served as Prime Minister?",
     "Three. He won general elections in 2014, in 2019 and again in 2024."),
    ("What is his constituency?",
     "He represents Varanasi in Uttar Pradesh in the Lok Sabha, the lower house of "
     "the Indian Parliament."),
    ("Name one policy he is known for.",
     "The Goods and Services Tax, introduced in 2017, replaced a large number of "
     "separate state and central taxes with a single tax."),
    ("What is the Swachh Bharat mission?",
     "It is a national sanitation campaign launched in 2014 with the goal of "
     "improving waste management and ending open defecation."),
    ("Who was Prime Minister before him?",
     "Manmohan Singh, who served two terms from 2004 to 2014."),
    ("Summarise everything I have asked so far.",
     "You have asked about Narendra Modi's age, party, birthplace, career before "
     "politics, his time as Chief Minister, his terms and constituency as Prime "
     "Minister, two of his policies, and who held the office before him."),
]

print(len(CONVERSATION), "turns,",
      sum(len(u) + len(a) for u, a in CONVERSATION), "characters of text")

In [ ]:
WINDOW_K = 3          # keep the last three turns
SUMMARY_BUDGET = 400  # characters; an ASSUMPTION, see the note below

sizes = [len(u) + len(a) for u, a in CONVERSATION]

buffer_per, window_per, summary_per = [], [], []
for i, (question, _) in enumerate(CONVERSATION):
    history = sum(sizes[:i])                      # everything said so far
    kept = sum(sizes[max(0, i - WINDOW_K):i])     # only the last k turns
    q = len(question)

    buffer_per.append(history + q)
    window_per.append(kept + q)
    # Nothing older than the window yet? Then there is nothing to summarise.
    summary_per.append((SUMMARY_BUDGET + kept if i > WINDOW_K else history) + q)

print(f"{'turn':>4} {'buffer':>9} {'window(3)':>10} {'summary':>9}")
for i in range(len(CONVERSATION)):
    print(f"{i+1:>4} {buffer_per[i]:>9} {window_per[i]:>10} {summary_per[i]:>9}")
print(f"{'TOTAL':>4} {sum(buffer_per):>9} {sum(window_per):>10} {sum(summary_per):>9}")

assert sum(buffer_per) == 8696
assert sum(window_per) == 4251
assert sum(summary_per) == 7451

Look at the buffer column climbing while the window column stays flat. That is
the whole story in one table.

Now the result that does **not** match the usual sales pitch for summary memory.

In [ ]:
b, w, s = sum(buffer_per), sum(window_per), sum(summary_per)
print(f"buffer  {b}")
print(f"window  {w}   ->  {b/w:.2f}x cheaper than buffer")
print(f"summary {s}   ->  {b/s:.2f}x cheaper than buffer")
print()
print("Summary memory saves almost nothing here - and it is MORE expensive")
print("than the plain three-turn window. Why? Arithmetic: a 400-character")
print(f"summary replacing {sum(sizes)} characters of history is not much of a saving.")

> **Be careful what these numbers are.** They are **characters**, not tokens - no
> offline tokeniser was assumed here, and the *shape* of the growth is identical
> in either unit. The buffer and window columns are exact counts of the text
> above. The summary column **assumes** a fixed 400-character summary; no summary
> was generated, because that needs a model call. It is a budget, not a
> measurement, and it also ignores the cost of the extra call it would take.

## Part D - Buffer memory is quadratic

The reason buffer memory is dangerous is not that it is big. It is that every
turn re-sends everything before it, so the total grows with the **square** of the
number of turns.

In [ ]:
def totals(conversation, k=WINDOW_K, budget=SUMMARY_BUDGET):
    sizes = [len(u) + len(a) for u, a in conversation]
    b = w = s = 0
    for i, (question, _) in enumerate(conversation):
        history = sum(sizes[:i])
        kept = sum(sizes[max(0, i - k):i])
        b += history + len(question)
        w += kept + len(question)
        s += (budget + kept if i > k else history) + len(question)
    return b, w, s


print(f"{'turns':>6} {'buffer':>10} {'window':>9} {'summary':>9} "
      f"{'buf/win':>9} {'buf/summ':>9}")
for reps in (1, 2, 4, 8, 16):
    convo = CONVERSATION * reps              # lengthen the conversation
    b, w, s = totals(convo)
    print(f"{len(convo):>6} {b:>10} {w:>9} {s:>9} {b/w:>8.2f}x {b/s:>8.2f}x")

In [ ]:
b12, w12, _ = totals(CONVERSATION * 1)
b192, w192, s192 = totals(CONVERSATION * 16)

print(f"16x the turns costs buffer memory {b192/b12:>6.1f}x more")
print(f"16x the turns costs a window      {w192/w12:>6.1f}x more")
print()
print("Quadratic against linear. 16 squared is 256; the rest is the turns")
print("getting slightly longer on average.")

assert b192 / b12 > 250      # quadratic
assert w192 / w12 < 25       # linear

## What to take away

- Swapping provider changes **2 of 7 lines** - the import and the class name.
- A provider class defines **2 of its 87** public members. The rest is shared.
- LLM API calls are **stateless**; memory is how an application works around it.
- **Buffer memory is quadratic.** 16x the turns cost **291x** the characters.
- **Summary memory saves almost nothing on a short conversation** (1.17x at 12
  turns) and needs a long one to earn back its extra model call (15.86x at 192).

## Exercises

1. Change `WINDOW_K` from 3 to 1, and to 8. At what window size does window
   memory stop being cheaper than summary memory? Does that depend on the length
   of the conversation?
2. `SUMMARY_BUDGET` is a guess. Find the budget at which summary memory exactly
   matches a three-turn window at 48 turns. Is that a realistic summary length?
3. Buffer memory's total is `sum over i of (history before turn i)`. Derive the
   closed form for a conversation of `n` turns each of average size `m`, and
   check it against the table.
4. Real billing is per token, and tokens are roughly 4 characters of English.
   Redo the 192-turn row in tokens, then in money at $3 per million input tokens.
   Which memory strategy would you ship?
5. Summary memory as modelled here ignores the extra model call per turn. Add it
   - assume the summariser reads the older history and writes 400 characters -
   and find the conversation length where summary memory is genuinely ahead.